### Simple agent with custom tools

In [27]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama

@tool
def calculator(expression: str) -> str:
    """A simple calculator that can add, subtract, multiply, or divide two numbers.
    Input should be a mathematical expression like '2 + 2' or '15 / 3'."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error calculating: {str(e)}"

@tool
def format_text(format_type: str, text: str) -> str:
    """Convert text to uppercase, lowercase, or titlecase."""

    if format_type == "uppercase":
        return text.upper()
    elif format_type == "lowercase":
        return text.lower()
    elif format_type == "titlecase":
        return text.title()
    else:
        return "Invalid format type"
        
# Create Tool objects for our functions
tools = [
    calculator, format_text
]

# TODO: Create the agent and executor
model = "llama3.2:1b"

llm = ChatOllama(
    model=model,
    temperature=0
)


agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt = """
    You are a helpful assistant. Use the available tools to perform tasks asked by user. 
    Give to the point answer. Do not reveal your thought process
    
    You MUST call the toolcall whenever you are asked to perform formatting or calcualations

    Do not just explain the operation.
    Return the exact result produced by the tool.
    """    
)

# Test with simple questions
test_questions = [
    "What is 25 + 63?", 
    "Can you convert 'hello world' to uppercase?",
    "Calculate 15 * 7", 
    "titlecase: langchain is awesome",
]

# TODO: Run the tests
for question in test_questions:
    print(f"\n===== Testing: {question} =====")
    result = agent.invoke({
        "messages": [ {"role": "user", "content": {question}}]
    })
    print(result["messages"][-1].content)
    


===== Testing: What is 25 + 63? =====
25 + 63 = 88

===== Testing: Can you convert 'hello world' to uppercase? =====
"HELLO WORLD"

===== Testing: Calculate 15 * 7 =====
The result of 15 * 7 is 105.

===== Testing: titlecase: langchain is awesome =====
Langchain Is Awesome
